# Model Training Notebook — Fixed & Complete
All issues resolved. Ready for model training after running all cells in order.

In [ ]:
# ── FIXED: Correct classification imports (not regression) ──────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

# Classification models (fixed from Regressor → Classifier)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
os.makedirs('artifacts', exist_ok=True)
os.makedirs('plots', exist_ok=True)

print('All libraries loaded successfully.')

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_csv('Palo Alto Networks.csv')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# ── FIXED: Drop constant/redundant columns ───────────────────────────────────
redundant = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
df.drop(columns=[c for c in redundant if c in df.columns], inplace=True)
print(f'Shape after dropping redundant cols: {df.shape}')

In [ ]:
# ── FIXED: Encode target FIRST (Yes→1, No→0) ────────────────────────────────
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
print('Target distribution after encoding:')
print(df['Attrition'].value_counts())

In [ ]:
# ── FIXED: Encode binary columns BEFORE feature engineering ──────────────────
# OverTime must be 0/1 for WorkloadStress flag to work correctly
df['OverTime'] = df['OverTime'].map({'Yes': 1, 'No': 0})
df['Gender']   = df['Gender'].map({'Male': 1, 'Female': 0})

print('Binary encoding done.')
print(df[['OverTime', 'Gender']].value_counts())

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────
# Income-to-experience ratio
df['IncomePerYearExp'] = df['MonthlyIncome'] / (df['TotalWorkingYears'] + 1)

# Promotion delay
df['PromotionDelay'] = df['YearsAtCompany'] - df['YearsSinceLastPromotion']

# Engagement composite
df['EngagementScore'] = (
    df['JobSatisfaction'] +
    df['EnvironmentSatisfaction'] +
    df['RelationshipSatisfaction']
) / 3

# FIXED: OverTime is now 0/1, so this flag works correctly
df['WorkloadStress'] = (
    (df['OverTime'] == 1) & (df['WorkLifeBalance'] <= 2)
).astype(int)

# Bonus features
df['CareerStagnation'] = df['YearsInCurrentRole'] / (df['YearsAtCompany'] + 1)
df['ManagerRisk']      = (
    (df['RelationshipSatisfaction'] <= 2) & (df['YearsWithCurrManager'] >= 3)
).astype(int)

print('Feature engineering done.')
print(df[['IncomePerYearExp','PromotionDelay','EngagementScore','WorkloadStress','CareerStagnation','ManagerRisk']].describe().round(2))

In [ ]:
# ── One-Hot Encode remaining categoricals ────────────────────────────────────
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Columns to one-hot encode: {cat_cols}')

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Convert booleans to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'Shape after encoding: {df.shape}')

In [ ]:
# ── FIXED: Define X and y (was missing before) ───────────────────────────────
X = df.drop('Attrition', axis=1)
y = df['Attrition']

feature_names = X.columns.tolist()
joblib.dump(feature_names, 'artifacts/feature_names.pkl')

print(f'X shape: {X.shape}')
print(f'y distribution: {dict(y.value_counts())}')

In [ ]:
# ── Step 1: Stratified Train-Test Split ──────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Train size : {X_train.shape[0]}')
print(f'Test  size : {X_test.shape[0]}')
print(f'Train class distribution: {dict(y_train.value_counts())}')

In [ ]:
# ── Step 2: Scale (fit on train only, transform both) ────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # only transform on test

joblib.dump(scaler, 'artifacts/scaler.pkl')
print('Scaler fitted and saved.')

In [ ]:
# ── Step 3: SMOTE (only on training data — NEVER on test) ────────────────────
print(f'Before SMOTE — Train class distribution: {dict(pd.Series(y_train).value_counts())}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f'After  SMOTE — Train class distribution: {dict(pd.Series(y_train_res).value_counts())}')
print(f'\nX_train after SMOTE : {X_train_res.shape}')
print(f'X_test  (unchanged) : {X_test_scaled.shape}')
print('\n✅ SMOTE complete. Pipeline order: Split → Scale → SMOTE  ✓')

In [ ]:
# ── Save preprocessed data ───────────────────────────────────────────────────
np.save('artifacts/X_train.npy', X_train_res)
np.save('artifacts/y_train.npy', y_train_res)
np.save('artifacts/X_test.npy',  X_test_scaled)
np.save('artifacts/y_test.npy',  y_test.values)

X_test.to_csv('artifacts/X_test_raw.csv', index=False)
y_test.to_csv('artifacts/y_test_raw.csv', index=False)

print('All artifacts saved to artifacts/')
print('\n🚀 Ready for model training!')

---
## Model Training starts here

In [ ]:
# ── Evaluation helper ────────────────────────────────────────────────────────
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    metrics = {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_te, y_pred),  4),
        'Precision': round(precision_score(y_te, y_pred), 4),
        'Recall'   : round(recall_score(y_te, y_pred),    4),
        'F1-Score' : round(f1_score(y_te, y_pred),        4),
        'ROC-AUC'  : round(roc_auc_score(y_te, y_proba),  4),
    }

    print(f'\n{"-"*50}')
    print(f'  {name}')
    print(f'{"-"*50}')
    for k, v in metrics.items():
        if k != 'Model':
            print(f'  {k:<12}: {v}')
    print(classification_report(y_te, y_pred, target_names=['No Attrition', 'Attrition']))
    return model, metrics, y_pred, y_proba

print('Helper ready.')

In [ ]:
# ── Model 1: Logistic Regression (Baseline) ──────────────────────────────────
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model, lr_metrics, lr_pred, lr_proba = evaluate_model(
    'Logistic Regression', lr, X_train_res, y_train_res, X_test_scaled, y_test
)

In [ ]:
# ── Model 2: Random Forest with Tuning ───────────────────────────────────────
rf_param_grid = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2'],
    'class_weight'     : ['balanced'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_param_grid, n_iter=30, scoring='roc_auc',
    cv=cv, n_jobs=-1, random_state=42, verbose=1
)
rf_search.fit(X_train_res, y_train_res)
print(f'Best RF params  : {rf_search.best_params_}')
print(f'Best CV ROC-AUC : {rf_search.best_score_:.4f}')

rf_model, rf_metrics, rf_pred, rf_proba = evaluate_model(
    'Random Forest', rf_search.best_estimator_, X_train_res, y_train_res, X_test_scaled, y_test
)

In [ ]:
# ── Model 3: XGBoost with Tuning ─────────────────────────────────────────────
neg = int((y_train_res == 0).sum())
pos = int((y_train_res == 1).sum())
spw = neg / pos   # scale_pos_weight (not needed since SMOTE balanced, but kept for safety)

xgb_param_grid = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [3, 5, 7],
    'learning_rate'   : [0.01, 0.05, 0.1, 0.2],
    'subsample'       : [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'reg_alpha'       : [0, 0.1, 0.5],
    'reg_lambda'      : [1, 1.5, 2],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(eval_metric='auc', random_state=42, n_jobs=-1),
    xgb_param_grid, n_iter=30, scoring='roc_auc',
    cv=cv, n_jobs=-1, random_state=42, verbose=1
)
xgb_search.fit(X_train_res, y_train_res)
print(f'Best XGB params  : {xgb_search.best_params_}')
print(f'Best CV ROC-AUC  : {xgb_search.best_score_:.4f}')

xgb_model, xgb_metrics, xgb_pred, xgb_proba = evaluate_model(
    'XGBoost', xgb_search.best_estimator_, X_train_res, y_train_res, X_test_scaled, y_test
)

In [ ]:
# ── Comparison Table ─────────────────────────────────────────────────────────
results_df = pd.DataFrame([lr_metrics, rf_metrics, xgb_metrics]).set_index('Model')
print('\n===== MODEL COMPARISON =====')
print(results_df.to_string())
results_df.style.highlight_max(axis=0, color='#d4edda')

In [ ]:
# ── Confusion Matrices ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pred) in zip(axes, [
    ('Logistic Regression', lr_pred),
    ('Random Forest',       rf_pred),
    ('XGBoost',             xgb_pred)
]):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, pred),
        display_labels=['No Attrition', 'Attrition']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')
plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC Curves ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
for (name, proba, auc), color in zip([
    ('Logistic Regression', lr_proba,  lr_metrics['ROC-AUC']),
    ('Random Forest',       rf_proba,  rf_metrics['ROC-AUC']),
    ('XGBoost',             xgb_proba, xgb_metrics['ROC-AUC']),
], ['#4878CF', '#6ACC65', '#D65F5F']):
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)
ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curve Comparison')
ax.legend(); ax.grid(alpha=0.3); sns.despine()
plt.tight_layout()
plt.savefig('plots/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save Best Model ───────────────────────────────────────────────────────────
best_name  = results_df['ROC-AUC'].idxmax()
best_model = {'Logistic Regression': lr_model,
              'Random Forest': rf_model,
              'XGBoost': xgb_model}[best_name]

joblib.dump(best_model, 'artifacts/best_model.pkl')
joblib.dump(results_df, 'artifacts/results_df.pkl')

print(f'Best model: {best_name}')
print(f'Saved to  : artifacts/best_model.pkl')
print(f'\nNext step : Run step3_risk_scoring_shap.ipynb')